# Sanity checks — `is_ultra_processed_or_sugary_beverage_heavy`

26.48% flagged (13,155 / ~49,688 products). Goal: identify false positive patterns.

In [10]:
import pandas as pd

df = pd.read_csv("./data/processed/products_flagged.csv")
flag_col = "is_ultra_processed_or_sugary_beverage_heavy"

flagged = df[df[flag_col]].copy()
not_flagged = df[~df[flag_col]].copy()

print(f"Total products:  {len(df):,}")
print(f"Flagged:         {len(flagged):,} ({len(flagged)/len(df)*100:.1f}%)")
print(f"Not flagged:     {len(not_flagged):,}")


Total products:  49,688
Flagged:         10,652 (21.4%)
Not flagged:     39,036


## 1. Breakdown by department — are unexpected departments heavily flagged?

In [11]:
dept_summary = (
    df.groupby("department")
    .agg(
        total=("product_id", "count"),
        flagged=(flag_col, "sum"),
    )
    .assign(pct=lambda x: (x["flagged"] / x["total"] * 100).round(1))
    .sort_values("pct", ascending=False)
)
print(dept_summary.to_string())


                 total  flagged   pct
department                           
breakfast         1115      959  86.0
snacks            6264     4730  75.5
frozen            4007     1778  44.4
bakery            1516      454  29.9
pantry            5371     1279  23.8
beverages         4365      959  22.0
international     1139       98   8.6
bulk                38        2   5.3
dairy eggs        3449      184   5.3
deli              1322       48   3.6
canned goods      2092       57   2.7
other              548       15   2.7
dry goods pasta   1858       48   2.6
meat seafood       907       13   1.4
alcohol           1054       14   1.3
produce           1684       14   0.8
babies            1081        0   0.0
household         3085        0   0.0
missing           1258        0   0.0
pets               972        0   0.0
personal care     6563        0   0.0


## 2. Top flagged aisles — spot obvious false-positive aisles

In [12]:
aisle_summary = (
    df.groupby(["department", "aisle"])
    .agg(
        total=("product_id", "count"),
        flagged=(flag_col, "sum"),
    )
    .assign(pct=lambda x: (x["flagged"] / x["total"] * 100).round(1))
    .sort_values("pct", ascending=False)
    .head(40)
)
print(aisle_summary.to_string())


                                             total  flagged    pct
department    aisle                                               
snacks        cookies cakes                    874      874  100.0
              mint gum                         168      168  100.0
frozen        frozen dessert                   112      112  100.0
              frozen pizza                     335      335  100.0
breakfast     breakfast bars pastries          173      173  100.0
snacks        ice cream toppings                85       85  100.0
              trail mix snack mix               69       69  100.0
              chips pretzels                   989      989  100.0
breakfast     cereal                           454      454  100.0
              hot cereal pancake mixes         303      303  100.0
frozen        ice cream ice                   1091     1091  100.0
snacks        candy chocolate                 1246     1246  100.0
pantry        salad dressing toppings          560      560  1

## 3. Which keywords are firing most? Decompose by keyword group

Map each flagged product back to the keyword that triggered it to find the noisiest terms.

In [16]:
import re

# Mirror the keyword groups from identify_products.py
KEYWORD_GROUPS = {
    "carbonated/sugary drinks": [
        "soda", "soft drink", "cola", "pop", "root beer", "ginger ale",
        "lemonade", "fruit punch", "sweet tea", "iced tea", "energy drink", "sports drink",
        "juice drink", "nectar", "sweetened", "sugar added",
    ],
    "snacks/candy/ice cream": [
        "chips", "crisps", "pretzels", "snack", "snack mix",
        "candy", "candies", "chocolate", "confectionery", "gummies", "gum", "lollipop",
        "ice cream", "frozen dessert", "sherbet", "gelato",
    ],
    "bread/baked goods/cereal": [
        "white bread", "buns", "rolls", "white tortillas",
        "cookies", "biscuits", "crackers",
        "pastry", "pastries", "donut", "doughnut", "cake", "brownie", "cupcake", "muffin", "cake mix",
        "cereal", "breakfast cereal", "granola bar", "snack bar",
    ],
    "spreads/prepared meals": [
        "margarine", "frosting", "mayonnaise", "ketchup", "bbq sauce", "ranch dressing", "salad dressing",
        "pizza", "frozen pizza", "pasta meal", "microwave meal", "ready meal", "prepared meal",
        "frozen pie", "pot pie", "instant soup", "cup soup", "ramen", "instant noodles",
        "instant dessert", "pudding mix",
    ],
    "processed meat": [
        "nuggets", "fish sticks", "chicken sticks",
        "hot dog", "hotdogs", # "sausage", "burger",  # too broad if you want to avoid flagging all sausages/burgers
    ],
}

def norm(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s/&+-]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

combined = (
    flagged["product_name"].astype(str) + " " +
    flagged["aisle"].astype(str) + " " +
    flagged["department"].astype(str)
).apply(norm)

hit_counts = {}
for group, kws in KEYWORD_GROUPS.items():
    pattern = re.compile(r"\b(?:%s)\b" % "|".join(re.escape(k) for k in kws), re.IGNORECASE)
    hit_counts[group] = combined.str.contains(pattern).sum()

hit_df = (
    pd.Series(hit_counts, name="products_matched")
    .sort_values(ascending=False)
    .to_frame()
    .assign(pct_of_flagged=lambda x: (x["products_matched"] / len(flagged) * 100).round(1))
)
print("Keyword group breakdown (flagged products):")
print(hit_df.to_string())


Keyword group breakdown (flagged products):
                          products_matched  pct_of_flagged
snacks/candy/ice cream                5483            51.5
bread/baked goods/cereal              3550            33.3
spreads/prepared meals                1347            12.6
carbonated/sugary drinks              1089            10.2
processed meat                         113             1.1


## 4. Suspiciously broad single keywords — sample products per term

Check the noisiest single words (e.g. "drink", "spread", "bread", "crackers") that likely fire on wholesome products.

In [14]:
SUSPECT_TERMS = [
    "drink", "spread", "bread", "crackers", "noodles", "pie",
    "wrap", "roll", "bun", "snack", "cereal", "pudding",
]

for term in SUSPECT_TERMS:
    pat = re.compile(r"\b%s\b" % re.escape(term), re.IGNORECASE)
    hits = flagged[flagged["product_name"].str.contains(pat, na=False)]
    print(f"\n--- '{term}' fires on {len(hits)} products ---")
    sample = hits[["product_name", "aisle", "department"]].head(8)
    print(sample.to_string(index=False))



--- 'drink' fires on 276 products ---
                                                      product_name                aisle department
                              Liquid Drink Mix Blackberry Lemonade    cocoa drink mixes  beverages
                                      Green Berry Rush Juice Drink        juice nectars  beverages
                                         Lemonade Flavor Drink Mix    cocoa drink mixes  beverages
                                      Orange Flavored Sports Drink energy sports drinks  beverages
                                           Fruit Punch Juice Drink        juice nectars  beverages
                                   Organic Berry Fuel Energy Drink energy sports drinks  beverages
                                                 Cherry Soft Drink          soft drinks  beverages
Refreshers Strawberry Lemonade Sparkling Green Coffee Energy Drink        juice nectars  beverages

--- 'spread' fires on 13 products ---
                             pr

## 5. Flagged products in "healthy" departments — clear false positives

Departments like `produce`, `meat seafood`, `dairy eggs`, `bulk`, `babies` shouldn't be heavily flagged.

In [15]:
LIKELY_HEALTHY_DEPTS = ["produce", "meat seafood", "dairy eggs", "bulk", "babies", "personal care", "household", "pets"]

fp_candidates = flagged[flagged["department"].isin(LIKELY_HEALTHY_DEPTS)]
print(f"Flagged products in 'likely healthy' departments: {len(fp_candidates)}")
print()

for dept, grp in fp_candidates.groupby("department"):
    print(f"\n=== {dept} ({len(grp)} flagged) ===")
    print(grp[["product_name", "aisle"]].head(15).to_string(index=False))


Flagged products in 'likely healthy' departments: 213


=== bulk (2 flagged) ===
                  product_name                        aisle
               Vegetable Chips bulk dried fruits vegetables
Naturally Sweet Plantain Chips bulk dried fruits vegetables

=== dairy eggs (184 flagged) ===
                                                       product_name                         aisle
                  Creamy Chocolate Sugar Free Powder Coffee Creamer                         cream
                                           Trix Cotton Candy Yogurt                        yogurt
                                            Maple Almond Snack Pack refrigerated pudding desserts
                                                      Chocolate Soy               soy lactosefree
                                      Colby Pepper Jack Snack Bites               packaged cheese
                                 Triple Zero Chocolate Greek Yogurt                        yogurt
                   